In [2]:
import sys
print(sys.executable)

/Users/maciejreluga/Desktop/CCCP/DE_&_ML_Projects_for_CV/files/Reddit_Conspiracy_Analysis/venv/bin/python


In [12]:
from datasets import load_dataset

dataset = load_dataset("mteb/reddit-clustering", split="test")
print(dataset)
print(dataset[0])

Generating test split: 100%|██████████| 25/25 [00:00<00:00, 80.35 examples/s]


Dataset({
    features: ['sentences', 'labels'],
    num_rows: 25
})
{'sentences': ["'Brisbane's Banksy' Anthony Lister found guilty of graffiti", 'Dick Smith enters voluntary administration due to bad sales', 'Flood Water Damage Restoration: Acquiring Proper Services', 'What is a real AUSSIE? by Peter Drew', '"3 days, 17 lines and 218 stations later the quest to reach every station in Melbourne is complete. If you can\'t travel the world, travel the metro"', 'Smoke from the huge bushfire that ravaged Yarloop and over 21 thousand hectares of farmland. As of this post the fire is still going strong.', 'Question about Centrelink and their app', 'Triple Js twitter this morning.', 'Confidence in Australia property market continues | Realestatecoulisse.com', 'The cult of the arsehole', 'NSW ALP boss Jamie Clements resigns amid pressure over sexual harassment claims', "Andrew Bolt's Channel Ten show The Bolt Report reportedly cancelled by network", 'NBN Co rejects FOI request for basic FTTN 

In [13]:
import pandas as pd

df = pd.DataFrame({
    'title': dataset[0]['sentences'],
    'subreddit': dataset[0]['labels']
})

print(df.shape)
print(df.head(10))
print(df['subreddit'].unique())

(7194, 2)
                                               title      subreddit
0  'Brisbane's Banksy' Anthony Lister found guilt...  australia.txt
1  Dick Smith enters voluntary administration due...  australia.txt
2  Flood Water Damage Restoration: Acquiring Prop...  australia.txt
3               What is a real AUSSIE? by Peter Drew  australia.txt
4  "3 days, 17 lines and 218 stations later the q...  australia.txt
5  Smoke from the huge bushfire that ravaged Yarl...  australia.txt
6            Question about Centrelink and their app  australia.txt
7                    Triple Js twitter this morning.  australia.txt
8  Confidence in Australia property market contin...  australia.txt
9                           The cult of the arsehole  australia.txt
<ArrowStringArray>
[   'australia.txt',       'Coffee.txt',        'apple.txt',
       'boston.txt',      'belgium.txt', 'Christianity.txt',
    'Anarchism.txt',     'collapse.txt',    'chemistry.txt',
       'brasil.txt',        'China.txt',

In [14]:
print(df['subreddit'].value_counts())

subreddit
Anarchism.txt       994
brasil.txt          912
China.txt           685
chemistry.txt       636
Coffee.txt          632
australia.txt       626
Christianity.txt    553
collapse.txt        550
boston.txt          505
apple.txt           489
belgium.txt         375
Advice.txt          237
Name: count, dtype: int64


In [17]:
conspiracy_subs = ['Anarchism.txt', 'collapse.txt', 'Christianity.txt']
mainstream_subs = ['Coffee.txt', 'australia.txt', 'Advice.txt', 'boston.txt', 'chemistry.txt']

def kategoryzuj(subreddit):
    if subreddit in conspiracy_subs:
        return 'conspiracy'
    elif subreddit in mainstream_subs:
        return 'mainstream'
    else:
        return 'other'

df['category'] = df['subreddit'].apply(kategoryzuj)

print(df['category'].value_counts())

category
mainstream    2636
other         2461
conspiracy    2097
Name: count, dtype: int64


In [18]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Test
test = "This is a terrible conspiracy and the government is lying to us!"
result = analyzer.polarity_scores(test)
print(result)

{'neg': 0.531, 'neu': 0.469, 'pos': 0.0, 'compound': -0.8805}


In [20]:
df['compound'] = df['title'].apply(
    lambda x: analyzer.polarity_scores(x)['compound']
)

print(df[['title', 'category', 'compound']].head(11))

                                                title    category  compound
0   'Brisbane's Banksy' Anthony Lister found guilt...  mainstream   -0.4215
1   Dick Smith enters voluntary administration due...  mainstream   -0.7783
2   Flood Water Damage Restoration: Acquiring Prop...  mainstream   -0.4939
3                What is a real AUSSIE? by Peter Drew  mainstream    0.0000
4   "3 days, 17 lines and 218 stations later the q...  mainstream    0.0258
5   Smoke from the huge bushfire that ravaged Yarl...  mainstream    0.4939
6             Question about Centrelink and their app  mainstream    0.0000
7                     Triple Js twitter this morning.  mainstream    0.0000
8   Confidence in Australia property market contin...  mainstream    0.5106
9                            The cult of the arsehole  mainstream    0.0000
10  NSW ALP boss Jamie Clements resigns amid press...  mainstream   -0.7906


In [21]:
print(df.groupby('category')['compound'].mean())

category
conspiracy   -0.058313
mainstream    0.035987
other        -0.003984
Name: compound, dtype: float64


In [23]:
import plotly.express as px

avg_sentiment = df.groupby('category')['compound'].mean().reset_index()

fig = px.bar(
    avg_sentiment,
    x='category',
    y='compound',
    color='category',
    title='Average Sentiment Score: Conspiracy vs Mainstream',
    labels={'compound': 'Average Sentiment (VADER)', 'category': 'Community Type'}
)

fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed